In [1]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    device_map = "cuda:0", # can only use 1 GPU.
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/rkuo/.agentic/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Gemma4 patching. Transformers: 5.14.1.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.478 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████████████████████████████████████████| 1951/1951 [00:00<00:00, 3643.23it/s]


In [2]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [3]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
)

In [4]:
from datasets import load_dataset
dataset = load_dataset("RinKana/finance-reasoning-synthetic", split="train")

Generating train split: 100%|███████████████████████████████| 1006/1006 [00:00<00:00, 55968.56 examples/s]


In [5]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

In [6]:
dataset[100]

{'question': 'How would you analyze antitrust concerns in a merger? What precedents would you review?',
 'cot': '<think>\n1.  **Antitrust Analysis:**\n    *   **Regulatory Bodies:** FTC/DOJ (USA), EC (Europe).\n2.  **Key Metrics:**\n    *   **HHI (Herfindahl-Hirschman Index):** Measure of market concentration. $\\sum s_i^2$.\n        *   Delta HHI > 200 in highly concentrated market -> Investigation.\n    *   **Market Definition:** Broad vs Narrow? (e.g., "Premium organic supermarkets" vs "All grocery stores").\n    *   **Remedies:** Asset sales (divestitures) to fix overlaps.\n3.  **Precedents:** Look at past deals in the sector. Did they get blocked?\n4.  **Draft Answer:**\n    *   HHI Index.\n    *   Market Definition.\n    *   Remedies analysis.\n</think>',
 'answer': "**Analysis Framework:**\\n1.  **HHI (Herfindahl-Hirschman Index):** Calculate the market concentration before and after the merger. A post-merger HHI increase of >200 points in a concentrated market typically trigger

In [7]:
def formatting_prompts_func(examples):
    # Extract the individual columns from your specific dataset
    questions = examples["question"]
    cots = examples["cot"]
    answers = examples["answer"]
    
    texts = []
    
    # Zip them together to iterate row by row
    for question, cot, answer in zip(questions, cots, answers):
        # Combine the chain-of-thought and the final answer for the assistant role
        assistant_content = f"{cot}\n{answer}"
        
        # Construct the conversation format with a default system prompt
        convo = [
            {"role": "system", "content": "You are an Expert financial Analyst."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": assistant_content}
        ]
        
        # Apply the chat template
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        texts.append(text)
        
    return { "text" : texts }

# Apply the mapping
dataset = dataset.map(formatting_prompts_func, batched = True)

Map: 100%|██████████████████████████████████████████████████| 1006/1006 [00:00<00:00, 21191.65 examples/s]


In [8]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=36): 100%|██████████████| 1006/1006 [00:56<00:00, 17.92 examples/s]


In [9]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map: 100%|███████████████████████████████████████████████████| 1006/1006 [00:00<00:00, 8274.31 examples/s]


In [10]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|turn>system\nYou are an Expert financial Analyst.<turn|>\n<|turn>user\nHow would you analyze antitrust concerns in a merger? What precedents would you review?<turn|>\n<|turn>model\n<think>\n1.  **Antitrust Analysis:**\n    *   **Regulatory Bodies:** FTC/DOJ (USA), EC (Europe).\n2.  **Key Metrics:**\n    *   **HHI (Herfindahl-Hirschman Index):** Measure of market concentration. $\\sum s_i^2$.\n        *   Delta HHI > 200 in highly concentrated market -> Investigation.\n    *   **Market Definition:** Broad vs Narrow? (e.g., "Premium organic supermarkets" vs "All grocery stores").\n    *   **Remedies:** Asset sales (divestitures) to fix overlaps.\n3.  **Precedents:** Look at past deals in the sector. Did they get blocked?\n4.  **Draft Answer:**\n    *   HHI Index.\n    *   Market Definition.\n    *   Remedies analysis.\n</think>\n**Analysis Framework:**\\n1.  **HHI (Herfindahl-Hirschman Index):** Calculate the market concentration before and after the merger. A post-merger HHI increase

In [11]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                    <think>\n1.  **Antitrust Analysis:**\n    *   **Regulatory Bodies:** FTC/DOJ (USA), EC (Europe).\n2.  **Key Metrics:**\n    *   **HHI (Herfindahl-Hirschman Index):** Measure of market concentration. $\\sum s_i^2$.\n        *   Delta HHI > 200 in highly concentrated market -> Investigation.\n    *   **Market Definition:** Broad vs Narrow? (e.g., "Premium organic supermarkets" vs "All grocery stores").\n    *   **Remedies:** Asset sales (divestitures) to fix overlaps.\n3.  **Precedents:** Look at past deals in the sector. Did they get blocked?\n4.  **Draft Answer:**\n    *   HHI Index.\n    *   Market Definition.\n    *   Remedies analysis.\n</think>\n**Analysis Framework:**\\n1.  **HHI (Herfindahl-Hirschman Index):** Calculate the market concentration before and after the merger. A post-merger HHI increase of >200 points in a concentrated market typically triggers an investigation.\\n2.  **Market Definition:** This is the key battleground. Regulators

In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5060 Ti. Max memory = 15.478 GB.
7.637 GB of memory reserved.


In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,006 | Num Epochs = 3 | Total steps = 756
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,079,104 of 5,116,376,608 (0.24% trained)


Step,Training Loss
1,0.684963
2,0.799841
3,0.762252
4,0.856882
5,0.787100
6,0.765927
7,0.842615
8,0.655957
9,0.801872
10,0.649230


In [14]:
messages = [
    {
        "role": "system",
        "content": [{"type" : "text", "text" : """You're financial analyst."""}]
    },
    {
        "role": "user",
        "content": [{"type" : "text", "text" : "A portfolio manager holds a long position in a stock and wants to hedge using put options. If the stock has a delta of 0.75 and a gamma of 0.05, what is the approximate delta of a 1-month put option (assume European style)?"}]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    enable_thinking = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 2048, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<|channel>thought
Here's a thinking process to arrive at the solution:

1.  **Analyze the Request:** The user is asking for the approximate delta of a 1-month European put option, given the delta and gamma of the underlying stock.
    *   Stock Delta ($\Delta_{stock}$): 0.75
    *   Stock Gamma ($\Gamma_{stock}$): 0.05
    *   Target: Put Option Delta ($\Delta_{put}$)

2.  **Identify the Relevant Relationship (Delta Hedging/Option Greeks):**
    *   The Delta of an option is typically calculated using the Black-Scholes model (or similar).
    *   The Delta of a Put option is related to the Delta of the Call option by Put-Call Parity, but more directly, it's related to the Delta of the underlying stock.

3.  **Recall the Delta Formula for European Options (Black-Scholes):**
    *   $\Delta_{Call} = N(d_1)$
    *   $\Delta_{Put} = N(d_1) - 1$ (or $N(d_1) - 1$ if $S$ is the stock price, $K$ is strike, $T$ is time, $r$ is rate, $\sigma$ is volatility).

4.  **Relate Stock Delta to Option D

In [15]:
#model.push_to_hub("RinKana/finance-gemma4-e2b-lora", token = hf_token)
#tokenizer.push_to_hub("RinKana/finance-gemma4-e2b-lora", token = hf_token)

In [16]:
#model.push_to_hub_merged(
#        "RinKana/finance-gemma4-e2b", tokenizer,
#        token = hf_token
#    )

In [17]:
model.save_pretrained("gemma_4_lora")
tokenizer.save_pretrained("gemma_4_lora")
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

Unsloth: Restored added_tokens_decoder metadata in gemma_4_lora/tokenizer_config.json.
